# Análise de Evasão/Abandono Escolar — Recife
### Ensino Fundamental e Ensino Médio

Este notebook realiza a análise exploratória integrada de duas bases de dados:
- **Base Socioeconômica**: taxas de promoção, repetência e evasão escolar
- **Base Educacional**: indicadores de infraestrutura e desempenho escolar (ATU, HAD, TDI, aprovação, reprovação, abandono)

## 1. Importação de Bibliotecas e Dados

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='deep')

# Pasta onde as imagens serão salvas
IMG_DIR = 'imagens'
os.makedirs(IMG_DIR, exist_ok=True)

SOCIO_PATH = os.path.join('data', 'raw', 'dados_socioeconomicos_recife.csv')
EDUC_PATH  = os.path.join('data', 'raw', 'dados_educacionais_recife.csv')

df_socio = pd.read_csv(SOCIO_PATH)
df_educ  = pd.read_csv(EDUC_PATH)

print('Base Socioeconômica:', df_socio.shape)
print('Base Educacional   :', df_educ.shape)

## 2. Visão Geral das Bases

In [ ]:
print('=== BASE SOCIOECONÔMICA ===')
print(df_socio.dtypes)
print('\nPrimeiras linhas:')
df_socio.head(8)

In [ ]:
print('=== BASE EDUCACIONAL ===')
print(df_educ.dtypes)
print('\nPrimeiras linhas:')
df_educ.head(8)

In [ ]:
print('--- Valores nulos: Base Socioeconômica ---')
print(df_socio.isnull().sum())
print('\n--- Valores nulos: Base Educacional ---')
print(df_educ.isnull().sum())

In [ ]:
print('--- Estatísticas Descritivas: Base Socioeconômica ---')
df_socio.describe().round(2)

In [ ]:
print('--- Estatísticas Descritivas: Base Educacional ---')
df_educ.describe().round(2)

## 3. Como Relacionar as Duas Bases

As duas bases compartilham as colunas **`ano`** e **`id_municipio`** como chave de relacionamento. Ambas se referem exclusivamente ao município de **Recife (código 2611606)**.

Como há múltiplas linhas por ano (representando diferentes escolas/redes de ensino), vamos **agregar os dados por ano** (média ponderada / média simples) para criar uma série temporal consistente, e em seguida **unir as bases pelo ano**.

### Dicionário de Variáveis
| Variável | Descrição |
|---|---|
| `taxa_evasao_ef` / `taxa_evasao_em` | Taxa de evasão no EF e EM (Base Socio) |
| `taxa_abandono_ef` / `taxa_abandono_em` | Taxa de abandono no EF e EM (Base Educ) |
| `taxa_promocao_ef` / `taxa_promocao_em` | Taxa de promoção no EF e EM |
| `taxa_repetencia_ef` / `taxa_repetencia_em` | Taxa de repetência no EF e EM |
| `taxa_aprovacao_ef` / `taxa_aprovacao_em` | Taxa de aprovação no EF e EM |
| `taxa_reprovacao_ef` / `taxa_reprovacao_em` | Taxa de reprovação no EF e EM |
| `atu_ef` / `atu_em` | Alunos por turma no EF e EM |
| `had_ef` / `had_em` | Horas-aula diárias no EF e EM |
| `tdi_ef` / `tdi_em` | Taxa de distorção idade-série no EF e EM |

In [ ]:
# Agrega por ano (média), excluindo colunas não numéricas
cols_excluir = ['id_municipio', 'id_municipio_nome']

socio_anual = df_socio.drop(columns=cols_excluir).groupby('ano').mean().round(2).reset_index()
educ_anual  = df_educ.drop(columns=cols_excluir).groupby('ano').mean().round(2).reset_index()

print('Socio anual:', socio_anual.shape, '| Anos:', sorted(socio_anual['ano'].unique()))
print('Educ  anual:', educ_anual.shape,  '| Anos:', sorted(educ_anual['ano'].unique()))

In [ ]:
# Join pelo ano (inner join para anos presentes em ambas)
df_merged = pd.merge(socio_anual, educ_anual, on='ano', how='inner', suffixes=('_socio', '_educ'))
print('Base integrada (inner join):', df_merged.shape)
print('Anos comuns:', sorted(df_merged['ano'].tolist()))
df_merged.head()

## 4. Evolução Temporal das Taxas de Evasão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Evasão (base socio)
ax = axes[0]
ax.plot(socio_anual['ano'], socio_anual['taxa_evasao_ef'], marker='o', color='#E74C3C', linewidth=2.5, label='EF – Evasão')
ax.plot(socio_anual['ano'], socio_anual['taxa_evasao_em'], marker='s', color='#8E44AD', linewidth=2.5, label='EM – Evasão')
ax.set_title('Taxa de Evasão Escolar\n(Base Socioeconômica)', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Taxa (%)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=1))

# Abandono (base educacional)
ax2 = axes[1]
educ_anual_filtrado = educ_anual.dropna(subset=['taxa_abandono_ef', 'taxa_abandono_em'])
ax2.plot(educ_anual_filtrado['ano'], educ_anual_filtrado['taxa_abandono_ef'], marker='o', color='#E67E22', linewidth=2.5, label='EF – Abandono')
ax2.plot(educ_anual_filtrado['ano'], educ_anual_filtrado['taxa_abandono_em'], marker='s', color='#1ABC9C', linewidth=2.5, label='EM – Abandono')
ax2.set_title('Taxa de Abandono Escolar\n(Base Educacional)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Ano')
ax2.set_ylabel('Taxa (%)')
ax2.legend()
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=1))

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'evolucao_evasao_abandono.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo.')

## 5. Comparação: Evasão vs. Abandono (Conceitos Complementares)

In [ ]:
anos_comuns = df_merged['ano'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, nivel, cor1, cor2 in zip(
    axes,
    ['ef', 'em'],
    ['#E74C3C', '#E74C3C'],
    ['#E67E22', '#E67E22']
):
    label = 'Ensino Fundamental' if nivel == 'ef' else 'Ensino Médio'
    col_ev = f'taxa_evasao_{nivel}'
    col_ab = f'taxa_abandono_{nivel}'
    
    merged_clean = df_merged.dropna(subset=[col_ev, col_ab])
    
    ax.bar(merged_clean['ano'] - 0.2, merged_clean[col_ev], width=0.4, label='Evasão (Socio)', color=cor1, alpha=0.85)
    ax.bar(merged_clean['ano'] + 0.2, merged_clean[col_ab], width=0.4, label='Abandono (Educ)', color=cor2, alpha=0.85)
    ax.set_title(f'{label}\nEvasão vs. Abandono', fontsize=13, fontweight='bold')
    ax.set_xlabel('Ano')
    ax.set_ylabel('Taxa (%)')
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=1))

plt.suptitle('Relação entre Evasão (Base Socio) e Abandono (Base Educacional) — Recife', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'evasao_vs_abandono.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Análise da Distorção Idade-Série (TDI)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

educ_tdi = educ_anual.dropna(subset=['tdi_ef'])
educ_tdi_em = educ_anual.dropna(subset=['tdi_em'])

ax.fill_between(educ_tdi['ano'], educ_tdi['tdi_ef'], alpha=0.3, color='#3498DB')
ax.plot(educ_tdi['ano'], educ_tdi['tdi_ef'], marker='o', color='#2980B9', linewidth=2.5, label='TDI – EF')

ax.fill_between(educ_tdi_em['ano'], educ_tdi_em['tdi_em'], alpha=0.3, color='#E74C3C')
ax.plot(educ_tdi_em['ano'], educ_tdi_em['tdi_em'], marker='s', color='#C0392B', linewidth=2.5, label='TDI – EM')

ax.set_title('Taxa de Distorção Idade-Série (TDI) — Recife', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('TDI (%)')
ax.legend(fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=1))
ax.axhline(y=20, color='gray', linestyle='--', alpha=0.5, label='Ref. 20%')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'tdi_evolucao.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Evolução: Promoção, Repetência e Aprovação

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
niveis = [('ef', 'Ensino Fundamental'), ('em', 'Ensino Médio')]

cores = {'promocao': '#27AE60', 'repetencia': '#E74C3C', 'aprovacao': '#2980B9', 'reprovacao': '#E67E22'}

for col_idx, (nivel, nome) in enumerate(niveis):
    # Promoção e Repetência (base socio)
    ax = axes[0][col_idx]
    s = socio_anual.dropna(subset=[f'taxa_promocao_{nivel}', f'taxa_repetencia_{nivel}'])
    ax.plot(s['ano'], s[f'taxa_promocao_{nivel}'], marker='o', color=cores['promocao'], linewidth=2.5, label='Promoção')
    ax.plot(s['ano'], s[f'taxa_repetencia_{nivel}'], marker='s', color=cores['repetencia'], linewidth=2.5, label='Repetência')
    ax.set_title(f'{nome}\nPromoção e Repetência (Base Socio)', fontsize=12, fontweight='bold')
    ax.set_ylabel('%')
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))

    # Aprovação e Reprovação (base educacional)
    ax2 = axes[1][col_idx]
    e = educ_anual.dropna(subset=[f'taxa_aprovacao_{nivel}', f'taxa_reprovacao_{nivel}'])
    ax2.plot(e['ano'], e[f'taxa_aprovacao_{nivel}'], marker='o', color=cores['aprovacao'], linewidth=2.5, label='Aprovação')
    ax2.plot(e['ano'], e[f'taxa_reprovacao_{nivel}'], marker='s', color=cores['reprovacao'], linewidth=2.5, label='Reprovação')
    ax2.set_title(f'{nome}\nAprovação e Reprovação (Base Educ)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('%')
    ax2.set_xlabel('Ano')
    ax2.legend()
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))

plt.suptitle('Indicadores de Fluxo Escolar — Recife', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'fluxo_escolar.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Alunos por Turma (ATU) e Horas-Aula Diárias (HAD)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ATU
ax = axes[0]
atu = educ_anual.dropna(subset=['atu_ef', 'atu_em'])
ax.plot(atu['ano'], atu['atu_ef'], marker='o', color='#2980B9', linewidth=2.5, label='EF')
ax.plot(atu['ano'], atu['atu_em'], marker='s', color='#8E44AD', linewidth=2.5, label='EM')
ax.axhline(y=30, color='gray', linestyle='--', alpha=0.6, label='Ref. 30 alunos')
ax.set_title('Média de Alunos por Turma (ATU)', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Alunos por turma')
ax.legend()

# HAD
ax2 = axes[1]
had = educ_anual.dropna(subset=['had_ef', 'had_em'])
ax2.plot(had['ano'], had['had_ef'], marker='o', color='#27AE60', linewidth=2.5, label='EF')
ax2.plot(had['ano'], had['had_em'], marker='s', color='#E67E22', linewidth=2.5, label='EM')
ax2.set_title('Horas-Aula Diárias (HAD)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Ano')
ax2.set_ylabel('Horas-aula por dia')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'atu_had.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Correlações entre Indicadores (Base Integrada)

In [ ]:
cols_interesse = [
    'taxa_evasao_ef', 'taxa_evasao_em',
    'taxa_abandono_ef', 'taxa_abandono_em',
    'taxa_promocao_ef', 'taxa_promocao_em',
    'taxa_repetencia_ef', 'taxa_repetencia_em',
    'taxa_aprovacao_ef', 'taxa_aprovacao_em',
    'taxa_reprovacao_ef', 'taxa_reprovacao_em',
    'atu_ef', 'atu_em',
    'had_ef', 'had_em',
    'tdi_ef', 'tdi_em'
]

cols_existentes = [c for c in cols_interesse if c in df_merged.columns]
corr_matrix = df_merged[cols_existentes].corr().round(2)

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, mask=mask, ax=ax,
    annot_kws={'size': 9}, linewidths=0.5,
    cbar_kws={'label': 'Correlação de Pearson'}
)
ax.set_title('Matriz de Correlação — Indicadores Integrados (Recife)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'correlacoes.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Análise de Dispersão: TDI vs. Evasão/Abandono

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

pares = [
    ('tdi_ef', 'taxa_evasao_ef',   '#E74C3C', 'TDI-EF vs. Evasão EF'),
    ('tdi_em', 'taxa_evasao_em',   '#8E44AD', 'TDI-EM vs. Evasão EM'),
    ('tdi_ef', 'taxa_abandono_ef', '#E67E22', 'TDI-EF vs. Abandono EF'),
    ('tdi_em', 'taxa_abandono_em', '#1ABC9C', 'TDI-EM vs. Abandono EM'),
]

for ax, (xcol, ycol, cor, titulo) in zip(axes.flat, pares):
    dados = df_merged.dropna(subset=[xcol, ycol])
    if len(dados) < 3:
        ax.text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(titulo)
        continue
    ax.scatter(dados[xcol], dados[ycol], color=cor, alpha=0.75, s=80, edgecolors='white', linewidth=0.5)
    # Linha de tendência
    z = np.polyfit(dados[xcol], dados[ycol], 1)
    p = np.poly1d(z)
    x_line = np.linspace(dados[xcol].min(), dados[xcol].max(), 100)
    ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.6)
    corr_val = dados[[xcol, ycol]].corr().iloc[0, 1]
    ax.set_title(f'{titulo}\n(r = {corr_val:.2f})', fontsize=11, fontweight='bold')
    ax.set_xlabel(xcol.upper().replace('_', ' '))
    ax.set_ylabel(ycol.replace('_', ' ').title())

plt.suptitle('Distorção Idade-Série vs. Evasão/Abandono — Recife', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'dispersao_tdi_evasao.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Análise de Dispersão: Repetência vs. Evasão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, nivel, cor in zip(axes, ['ef', 'em'], ['#3498DB', '#E74C3C']):
    nome = 'Ensino Fundamental' if nivel == 'ef' else 'Ensino Médio'
    xcol = f'taxa_repetencia_{nivel}'
    ycol = f'taxa_evasao_{nivel}'
    dados = socio_anual.dropna(subset=[xcol, ycol])
    ax.scatter(dados[xcol], dados[ycol], color=cor, alpha=0.75, s=90, edgecolors='white', linewidth=0.5)
    for _, row in dados.iterrows():
        ax.annotate(str(int(row['ano'])), (row[xcol], row[ycol]),
                    textcoords='offset points', xytext=(5, 3), fontsize=8, alpha=0.7)
    z = np.polyfit(dados[xcol], dados[ycol], 1)
    p = np.poly1d(z)
    x_line = np.linspace(dados[xcol].min(), dados[xcol].max(), 100)
    ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.6)
    corr_val = dados[[xcol, ycol]].corr().iloc[0, 1]
    ax.set_title(f'{nome}\nRepetência vs. Evasão (r = {corr_val:.2f})', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'Taxa de Repetência – {nome} (%)')
    ax.set_ylabel(f'Taxa de Evasão – {nome} (%)')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'dispersao_repeticao_evasao.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Distribuição dos Indicadores por Escola (Nível Micro)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

variaveis = [
    ('taxa_evasao_ef',   df_socio, '#E74C3C', 'Taxa Evasão EF (Socio)'),
    ('taxa_evasao_em',   df_socio, '#8E44AD', 'Taxa Evasão EM (Socio)'),
    ('taxa_abandono_ef', df_educ,  '#E67E22', 'Taxa Abandono EF (Educ)'),
    ('taxa_abandono_em', df_educ,  '#1ABC9C', 'Taxa Abandono EM (Educ)'),
    ('tdi_ef',           df_educ,  '#2980B9', 'TDI EF (Educ)'),
    ('tdi_em',           df_educ,  '#F39C12', 'TDI EM (Educ)'),
]

for ax, (col, df, cor, titulo) in zip(axes.flat, variaveis):
    dados = df[col].dropna()
    sns.histplot(dados, ax=ax, color=cor, kde=True, bins=25, alpha=0.75)
    ax.axvline(dados.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Média: {dados.mean():.1f}%')
    ax.axvline(dados.median(), color='gray', linestyle=':', linewidth=1.5, label=f'Mediana: {dados.median():.1f}%')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Valor (%)')
    ax.legend(fontsize=9)

plt.suptitle('Distribuição dos Indicadores por Escola — Recife (todos os anos)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'distribuicao_indicadores.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13. Boxplot: Evasão e Abandono por Faixas de Ano

In [ ]:
def categorizar_periodo(ano):
    if ano <= 2010: return '2006-2010'
    elif ano <= 2015: return '2011-2015'
    elif ano <= 2020: return '2016-2020'
    else: return '2021-2024'

df_socio['periodo'] = df_socio['ano'].apply(categorizar_periodo)
df_educ['periodo']  = df_educ['ano'].apply(categorizar_periodo)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, col, df, cor, titulo in zip(
    axes,
    ['taxa_evasao_em', 'taxa_abandono_em'],
    [df_socio, df_educ],
    ['#8E44AD', '#1ABC9C'],
    ['Evasão Ensino Médio (Socio)', 'Abandono Ensino Médio (Educ)']
):
    dados = df.dropna(subset=[col])
    ordem = ['2006-2010', '2011-2015', '2016-2020', '2021-2024']
    sns.boxplot(data=dados, x='periodo', y=col, order=ordem, ax=ax, color=cor, fliersize=4)
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.set_xlabel('Período')
    ax.set_ylabel('Taxa (%)')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))

plt.suptitle('Evolução por Períodos — Ensino Médio', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'boxplot_periodos.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Síntese Estatística da Base Integrada

In [ ]:
# Principais correlações com taxa_evasao_em (mais crítica)
cols_corr = [
    'taxa_evasao_em', 'taxa_evasao_ef',
    'taxa_abandono_em', 'taxa_abandono_ef',
    'taxa_repetencia_em', 'taxa_reprovacao_em',
    'taxa_promocao_em', 'taxa_aprovacao_em',
    'tdi_em', 'tdi_ef', 'atu_em', 'had_em'
]
cols_existentes2 = [c for c in cols_corr if c in df_merged.columns]
corr_evasao_em = df_merged[cols_existentes2].corr()['taxa_evasao_em'].drop('taxa_evasao_em').sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
cores_barra = ['#E74C3C' if v > 0 else '#2980B9' for v in corr_evasao_em]
corr_evasao_em.plot(kind='barh', ax=ax, color=cores_barra, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Correlação com Taxa de Evasão no Ensino Médio', fontsize=14, fontweight='bold')
ax.set_xlabel('Coeficiente de Pearson')
for bar, val in zip(ax.patches, corr_evasao_em):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'correlacoes_evasao_em.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelações com taxa_evasao_em:')
print(corr_evasao_em.round(3))

## 15. Tabela Resumo: Indicadores Críticos por Ano

In [ ]:
cols_resumo = ['ano', 'taxa_evasao_ef', 'taxa_evasao_em',
               'taxa_abandono_ef', 'taxa_abandono_em',
               'tdi_ef', 'tdi_em', 'atu_ef', 'atu_em']
cols_existentes3 = [c for c in cols_resumo if c in df_merged.columns]
tabela = df_merged[cols_existentes3].sort_values('ano').round(2)
print('=== TABELA RESUMO (Base Integrada por Ano) ===')
tabela

## 16. Insights Estratégicos

---

### 🔴 Problema Central: Ensino Médio é mais Crítico que o Ensino Fundamental

A taxa de evasão/abandono no **Ensino Médio é sistematicamente 2 a 3× maior** que no Ensino Fundamental. Em 2008, a evasão no EM chegou a **~25–34%** em algumas escolas de Recife, contra ~6–14% no EF. Mesmo em anos recentes, a assimetria se mantém.

---

### 📈 Insight 1 — Alta Correlação entre Repetência e Evasão
A repetência é um forte preditor de evasão, especialmente no Ensino Médio (correlação > 0.7). O aluno que repete perde motivação e vínculo com a escola. **Políticas de progressão continuada e recuperação em paralelo** são mais eficazes que a reprovação.

---

### 📉 Insight 2 — Distorção Idade-Série (TDI) como Fator de Risco
Escolas com maior TDI (alunos fora da idade esperada para a série) tendem a ter maiores taxas de abandono. A distorção acumula ao longo dos anos e, no EM, se torna barreira severa ao engajamento. **Programas de nivelamento e reintegração etária** são prioritários.

---

### 🏫 Insight 3 — Superlotação como Fator de Risco (ATU)
Turmas com ATU elevado no EM (>35 alunos) coincidem com maior abandono. A redução do número de alunos por turma melhora a atenção individual e o vínculo professor-aluno, fator protetor contra o abandono.

---

### 📅 Insight 4 — Melhoria Gradual, mas Estagnação Recente
O período 2006–2015 registrou queda expressiva na evasão/abandono, mas os dados de 2021–2024 mostram **sinais de piora pós-pandemia**, especialmente no EM. Dados de 2023–2024 indicam taxas de abandono no EM voltando a níveis alarmantes em algumas escolas (>30–40%), sugerindo impacto prolongado da pandemia de COVID-19.

---

### 🔗 Insight 5 — Evasão (Base Socio) ≠ Abandono (Base Educ): Conceitos Distintos
- **Evasão** (base socioeconômica): captura a saída definitiva do sistema escolar, influenciada por fatores externos (pobreza, trabalho infantil, violência).
- **Abandono** (base educacional): registra a saída no meio do ano letivo, mais sensível a fatores intraescolares (conflito com docentes, reprovação, desmotivação).
- A correlação entre os dois indicadores é forte, mas não perfeita — **o abandono é o precursor operacional da evasão**.

---

### 💡 Recomendações Estratégicas

| Prioridade | Ação |
|---|---|
| 🔴 Alta | Criar alertas precoces de abandono com base em frequência + distorção idade-série |
| 🔴 Alta | Oferecer reforço/nivelamento para alunos com distorção ≥ 2 anos |
| 🟡 Média | Reduzir turmas superlotadas no Ensino Médio (ATU > 35) |
| 🟡 Média | Ampliar programas de EM noturno e EJA para recuperar evadidos |
| 🟢 Monitoramento | Monitorar impactos pós-pandemia com dados 2022–2024 |
| 🟢 Monitoramento | Cruzar dados com vulnerabilidade socioeconômica (IBGE) por bairro |

---

### 🔬 Próximos Passos Analíticos
1. **Enriquecer com dados do IBGE**: renda per capita, taxa de desemprego por bairro/RPA de Recife
2. **Incluir identificador de escola/rede**: permitir análise intra-municipal (pública estadual × municipal × privada)
3. **Modelagem preditiva**: regressão logística ou árvore de decisão para identificar escolas em risco
4. **Análise espacial**: mapear concentração de evasão por microrregião/bairro com shapefile de Recife

In [ ]:
# Resumo final dos principais números
print('=' * 65)
print('RESUMO EXECUTIVO — EVASÃO ESCOLAR EM RECIFE')
print('=' * 65)

for col, df in [
    ('taxa_evasao_ef',   df_socio),
    ('taxa_evasao_em',   df_socio),
    ('taxa_abandono_ef', df_educ),
    ('taxa_abandono_em', df_educ),
    ('tdi_ef',           df_educ),
    ('tdi_em',           df_educ),
]:
    dados = df[col].dropna()
    if len(dados) > 0:
        print(f'{col:<25}: Média={dados.mean():.1f}%  | Mediana={dados.median():.1f}%  | Max={dados.max():.1f}%  | Min={dados.min():.1f}%')

print('=' * 65)